In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error


In [2]:
hierarchy = pd.read_csv("company_hierarchy.csv")
emp_df = pd.read_csv("employee.csv")


In [3]:
df = emp_df.merge(hierarchy, on='employee_id', how='left')
print(df.head(5))

   employee_id  signing_bonus    salary degree_level sex  yrs_experience  \
0       138719              0  273000.0       Master   M               2   
1         3192              0  301000.0     Bachelor   F               1   
2       114657              0  261000.0       Master   F               2   
3        29039              0   86000.0  High_School   F               4   
4       118607              0  126000.0     Bachelor   F               3   

    boss_id         dept  
0   43602.0  engineering  
1   87847.0        sales  
2  180854.0        sales  
3   88370.0           HR  
4   23565.0        sales  


In [4]:
employee_tree = defaultdict(list)

for _, row in hierarchy.iterrows():
    employee_id = row["employee_id"]
    boss_id = row["boss_id"]

    if pd.notna(boss_id):
        employee_tree[boss_id].append(employee_id)

employee_tree

defaultdict(list,
            {175361.0: [46456,
              25464,
              70036,
              132143,
              155125,
              79354,
              184993,
              155427,
              77361,
              42617,
              41769,
              63694,
              13430,
              91050,
              197373,
              43707,
              115208],
             29733.0: [104708,
              109610,
              59052,
              160686,
              45200,
              117246,
              165229,
              87889,
              194720,
              81227,
              32288],
             41991.0: [120853,
              127766,
              137105,
              42756,
              85081,
              171182,
              101952,
              196156,
              124824,
              44544,
              30087,
              101095],
             171266.0: [142630, 139278, 8812, 75705, 7095],
             198240.0: [72711,


In [5]:
ceo = hierarchy[hierarchy["boss_id"].isna()]["employee_id"].iloc[0]
print(ceo)

61554


In [6]:
# Start everyone as IC
df["level"] = "IC"

# CEO
df.loc[df["employee_id"] == ceo, "level"] = "CEO"

# Executives: boss is CEO
e_ids = df.loc[df["boss_id"] == ceo, "employee_id"]
df.loc[df["employee_id"].isin(e_ids), "level"] = "E"

# Vice Presidents: boss is an Executive
vp_ids = df.loc[df["boss_id"].isin(e_ids), "employee_id"]
df.loc[df["employee_id"].isin(vp_ids), "level"] = "VP"

# Directors: boss is a VP
d_ids = df.loc[df["boss_id"].isin(vp_ids), "employee_id"]
df.loc[df["employee_id"].isin(d_ids), "level"] = "D"

# Middle Managers: boss is a Director
mm_ids = df.loc[df["boss_id"].isin(d_ids), "employee_id"]
df.loc[df["employee_id"].isin(mm_ids), "level"] = "MM"

# Individual Contributors: boss is a Middle Manager
ic_ids = df.loc[df["boss_id"].isin(mm_ids), "employee_id"]
df.loc[df["employee_id"].isin(ic_ids), "level"] = "IC"



In [7]:
print(df["level"].value_counts())

level
IC     9000
MM      800
D       160
VP       35
E         4
CEO       1
Name: count, dtype: int64


In [8]:
df.groupby("level")["salary"].mean().sort_values()

level
IC     187909.222222
MM     192473.750000
D      212343.750000
VP     258028.571429
E      562500.000000
CEO    700000.000000
Name: salary, dtype: float64

In [9]:
df.groupby("sex")["salary"].mean()

sex
F    171314.518394
M    198954.340736
Name: salary, dtype: float64

In [10]:
df.groupby(["level", "sex"])["salary"].mean()

level  sex
CEO    M      700000.000000
D      F      209085.106383
       M      213699.115044
E      F      500000.000000
       M      583333.333333
IC     F      169473.456790
       M      198279.340278
MM     F      182935.606061
       M      197171.641791
VP     F      259444.444444
       M      257538.461538
Name: salary, dtype: float64

In [11]:
features = [
    "yrs_experience",
    "signing_bonus",
    "sex",
    "degree_level",
    "dept",
    "level"
]

X = df[features]
y = df["salary"]

In [12]:
X = pd.get_dummies(X, drop_first=True)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [15]:
predictions = model.predict(X_test)

In [16]:
r2 = r2_score(y_test, predictions)

print("R²:", r2)

R²: 0.3553938940921456


In [17]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("RMSE:", rmse)

RMSE: 70619.83038469568


In [18]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coefficients.sort_values(by="Coefficient", ascending=False)

,Feature,Coefficient
11,level_E,92901.124652
5,degree_level_PhD,724.643170
0,yrs_experience,594.529367
2,sex_M,-1863.084175
4,degree_level_Master,-2011.730859
1,signing_bonus,-2721.527479
3,degree_level_High_School,-3114.141432
7,dept_engineering,-172642.705801
14,level_VP,-215028.090164
9,dept_sales,-222530.526578
